In [38]:
from google.colab import files

uploaded = files.upload()

Saving energy_master.csv to energy_master (1).csv


In [39]:
df = energy[['price', 'load', 'wind', 'solar']].copy()

df = df.dropna()
df['price_lag_1'] = df['price'].shift(1)
df['hour'] = df.index.hour

df = df.dropna()

In [40]:
#STRATEGY INTERFACE
class Strategy:
    def get_signal(self, row):
        pass

In [41]:
class MeanReversionStrategy(Strategy):

    def get_signal(self, row):

        if row['price'] < row['price_lag_1']:
            return 1   # charge
        elif row['price'] > row['price_lag_1']:
            return -1  # discharge
        else:
            return 0

In [43]:
#BATTERY SIMULATOR
class Battery:
    def __init__(self, capacity=100, rate=20):
        self.capacity = capacity
        self.rate = rate
        self.level = 50

    def step(self, action):
        if action == 1:
            self.level = min(self.capacity, self.level + self.rate)

        elif action == -1:
            self.level = max(0, self.level - self.rate)

In [44]:
#BACKTEST ENGINE
def backtest(df, strategy):

    battery = Battery()

    cash = 0
    positions = []

    for i in range(len(df)):

        row = df.iloc[i]

        action = strategy.get_signal(row)

        price = row['price']

        # execute
        if action == 1:
            cash -= price * battery.rate
            battery.step(1)

        elif action == -1:
            cash += price * battery.rate
            battery.step(-1)

        positions.append(battery.level)

    return cash, positions

In [45]:
#METRICS ENGINE
def metrics(pnl_series):

    pnl = np.array(pnl_series)

    return {
        "total_return": pnl[-1],
        "mean": pnl.mean(),
        "volatility": pnl.std(),
        "sharpe": pnl.mean() / pnl.std()
    }

In [46]:
strategy = MeanReversionStrategy()

pnl, positions = backtest(df, strategy)

print("Final PnL:", pnl)

Final PnL: -730307.8000000004


In [47]:
returns = df[['price', 'wind', 'solar', 'load']].pct_change().dropna()

In [48]:
#MEAN & COV MATRIX
mu = returns.mean()
cov = returns.cov()

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:127: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2767: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]


In [49]:
#MARKOWITZ PORTFOLIO
import numpy as np
from scipy.optimize import minimize

In [50]:
def portfolio_variance(w, cov):
    return w.T @ cov @ w


In [51]:
def constraint_sum(w):
    return np.sum(w) - 1

In [52]:
def constraint_return(w, mu, target):
    return w @ mu - target

In [53]:
n = len(mu)
w0 = np.ones(n) / n

constraints = [
    {"type": "eq", "fun": constraint_sum},
    {"type": "eq", "fun": lambda w: constraint_return(w, mu, 0.001)}
]

bounds = [(0, 1)] * n

res = minimize(
    portfolio_variance,
    w0,
    args=(cov,),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

weights = res.x

In [54]:
#EFFICIENT FRONTIER
returns_range = np.linspace(0.0001, 0.005, 50)

frontier_risks = []

for r in returns_range:

    constraints = [
        {"type": "eq", "fun": constraint_sum},
        {"type": "eq", "fun": lambda w: constraint_return(w, mu, r)}
    ]

    res = minimize(portfolio_variance, w0, args=(cov,),
                   method="SLSQP", bounds=bounds,
                   constraints=constraints)

    frontier_risks.append(res.fun)

In [55]:
#CVaR OPTIMIZATION
losses = -returns

In [56]:
#CVaR approximation:
def cvar(w, losses, alpha=0.95):

    portfolio_loss = losses @ w

    var = np.quantile(portfolio_loss, alpha)

    cvar = portfolio_loss[portfolio_loss >= var].mean()

    return cvar

In [57]:
#Minimize CVaR:
res = minimize(
    cvar,
    w0,
    args=(losses,),
    method="SLSQP",
    bounds=bounds,
    constraints=[{"type": "eq", "fun": constraint_sum}]
)

cvar_weights = res.x

/usr/local/lib/python3.12/dist-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


In [59]:
print(cvar_weights)
print(cvar)

[0.25 0.25 0.25 0.25]
<function cvar at 0x7f0f2b340ea0>
